In [1]:
# Project-relative paths (supports notebooks launched from the project root or notebooks/).
from pathlib import Path
import sys

_PROJECT_CANDIDATES = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (p for p in _PROJECT_CANDIDATES if (p / "data" / "raw" / "mongo_test.test-result.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Launch Jupyter from the project root or notebooks/ directory.")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "artifacts" / "models"
PREPROCESSING_DIR = PROJECT_ROOT / "artifacts" / "preprocessing"
SPLITS_DIR = PROJECT_ROOT / "artifacts" / "splits"
for _path in (DATA_PROCESSED, MODELS_DIR, PREPROCESSING_DIR, SPLITS_DIR):
    _path.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))


In [2]:
import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

In [3]:
# ──────────────────────────────────────────────────────────
# 1. LOAD DATA
# ──────────────────────────────────────────────────────────

def load_data(path: str) -> pd.DataFrame:
    """
    Load from JSON array file.
    Adjust this function if your data is CSV or JSON Lines.
    """
    with open(path, "r") as f:
        records = json.load(f)
    return pd.DataFrame(records)

In [4]:
# ──────────────────────────────────────────────────────────
# 2. CLEAN & TYPE-FIX
# ──────────────────────────────────────────────────────────

def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["batchSize"] = df["batchSize"].astype(int)
    return df

In [5]:
# ──────────────────────────────────────────────────────────
# 3. SPLIT FEATURES / TARGETS
# ──────────────────────────────────────────────────────────

NUMERICAL_FEATURES   = ["batchSize", "lingerMs", "billSize"]
CATEGORICAL_FEATURES = ["compressionType"]
FEATURES             = NUMERICAL_FEATURES + CATEGORICAL_FEATURES
TARGETS              = ["throughput"]

# ── Constants ─────────────────────────
BATCH_SIZE_VALUES  = [16384, 32768, 49152, 65536]
LINGER_MS_VALUES   = list(range(5, 506, 20))
COMPRESSION_VALUES = ["none", "snappy", "lz4", "gzip"]

def split_xy(df: pd.DataFrame):
    X = df[FEATURES]
    y = df[TARGETS]
    return X, y


In [6]:
# 4. BUILD SKLEARN PREPROCESSOR
# ──────────────────────────────────────────────────────────

def build_preprocessor() -> ColumnTransformer:
    """
    - Numerical columns  → StandardScaler  (zero mean, unit variance)
    - compressionType    → OneHotEncoder   (none, snappy, lz4, gzip)

    Note for GPR: also normalize targets separately (see normalize_targets).
    """
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUMERICAL_FEATURES),
            ("cat", OneHotEncoder(
                categories=[["none", "snappy", "lz4", "gzip"]],
                sparse_output=False,
                handle_unknown="ignore"
            ), CATEGORICAL_FEATURES),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


def get_feature_names(preprocessor: ColumnTransformer) -> list[str]:
    num_names = NUMERICAL_FEATURES
    cat_names = list(
        preprocessor
        .named_transformers_["cat"]
        .get_feature_names_out(CATEGORICAL_FEATURES)
    )
    return num_names + cat_names



In [7]:
# ──────────────────────────────────────────────────────────
# 5. TARGET NORMALIZATION  (needed for GPR)
# ──────────────────────────────────────────────────────────

class TargetScaler:
    """
    Thin wrapper so we can inverse-transform predictions easily.
    Fits one StandardScaler per target column.
    """
    def __init__(self):
        self.scalers = {t: StandardScaler() for t in TARGETS}

    def fit_transform(self, y: pd.DataFrame) -> np.ndarray:
        cols = []
        for t in TARGETS:
            scaled = self.scalers[t].fit_transform(y[[t]])
            cols.append(scaled)
        return np.hstack(cols)

    def transform(self, y: pd.DataFrame) -> np.ndarray:
        cols = []
        for t in TARGETS:
            scaled = self.scalers[t].transform(y[[t]])
            cols.append(scaled)
        return np.hstack(cols)

    def inverse_transform(self, y_scaled: np.ndarray) -> pd.DataFrame:
        cols = {}
        for i, t in enumerate(TARGETS):
            cols[t] = self.scalers[t].inverse_transform(
                y_scaled[:, i].reshape(-1, 1)
            ).flatten()
        return pd.DataFrame(cols)


In [8]:
# ──────────────────────────────────────────────────────────
# 6. MAIN PIPELINE
# ──────────────────────────────────────────────────────────

def preprocess(path: str, test_size: float = 0.2, random_state: int = 42):
    """
    Full preprocessing pipeline.

    Returns
    -------
    X_train, X_test : np.ndarray  — scaled feature matrices
    y_train, y_test : pd.DataFrame — raw targets  (for RF / XGBoost)
    y_train_s, y_test_s : np.ndarray — scaled targets (for GPR)
    feature_names : list[str]
    preprocessor : fitted ColumnTransformer
    target_scaler : fitted TargetScaler
    """

    # Load & clean
    df = load_data(path)
    df = clean(df)

    # Describe the dataset
    print("=" * 55)
    print(f"Dataset shape    : {df.shape}")
    print(f"\nFeatures used    : {FEATURES}")
    print(f"Targets          : {TARGETS}")
    print(f"\ncompression_type distribution:")
    print(df["compressionType"].value_counts().to_string())
    print(f"\nbatchSize unique : {sorted(df['batchSize'].unique())}")
    print(f"lingerMs range   : {df['lingerMs'].min()} – {df['lingerMs'].max()}")
    print(f"billSize unique  : {sorted(df['billSize'].unique())}")
    print(f"\nThroughput stats :\n{df['throughput'].describe().round(2).to_string()}")
    print("=" * 55)

    # Features / targets
    X, y = split_xy(df)

    # Train / test split  (stratify not used — continuous targets)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # Fit preprocessor on train, apply to both splits
    preprocessor = build_preprocessor()
    X_train_s = preprocessor.fit_transform(X_train)
    X_test_s  = preprocessor.transform(X_test)

    feature_names = get_feature_names(preprocessor)
    print(f"\nFinal feature names ({len(feature_names)}):")
    print(feature_names)
    print(f"\nX_train : {X_train_s.shape}  |  X_test : {X_test_s.shape}")
    print(f"y_train : {y_train.shape}   |  y_test : {y_test.shape}")

    # Target scaling  (for GPR)
    target_scaler = TargetScaler()
    y_train_s = target_scaler.fit_transform(y_train)
    y_test_s  = target_scaler.transform(y_test)

    return (
        X_train_s, X_test_s,
        y_train, y_test,
        y_train_s, y_test_s,
        feature_names,
        preprocessor,
        target_scaler,
    )

In [9]:
results = preprocess(str(DATA_RAW / "mongo_test.test-result.json"))

Dataset shape    : (1248, 5)

Features used    : ['batchSize', 'lingerMs', 'billSize', 'compressionType']
Targets          : ['throughput']

compression_type distribution:
compressionType
none      312
lz4       312
snappy    312
gzip      312

batchSize unique : [np.int64(16384), np.int64(32768), np.int64(49152), np.int64(65536)]
lingerMs range   : 5 – 505
billSize unique  : [np.int64(309000), np.int64(618000), np.int64(927000)]

Throughput stats :
count      1248.00
mean      93217.72
std       26954.28
min       53636.52
25%       70158.18
50%       89263.39
75%      106286.01
max      187841.95

Final feature names (7):
['batchSize', 'lingerMs', 'billSize', 'compressionType_none', 'compressionType_snappy', 'compressionType_lz4', 'compressionType_gzip']

X_train : (998, 7)  |  X_test : (250, 7)
y_train : (998, 1)   |  y_test : (250, 1)


In [10]:
import json

# Reload and clean the original file
df_all = load_data(str(DATA_RAW / "mongo_test.test-result.json"))
df_all = clean(df_all)

# Export to JSON
df_all.to_json(str(DATA_PROCESSED / "cleaned_data.json"), orient="records", indent=2)
print("Saved!")

Saved!


In [11]:
(
    X_train_s, X_test_s,
    y_train, y_test,
    y_train_s, y_test_s,
    feature_names,
    preprocessor,
    target_scaler,
) = results

In [12]:
df_scaled = pd.DataFrame(X_train_s, columns=feature_names)
df_scaled.to_json(str(DATA_PROCESSED / "scaled_data.json"), orient="records", indent=2)

In [13]:
from lightgbm import LGBMRegressor

In [14]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [15]:
lgbm_model = LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

lgbm_model.fit(
    pd.DataFrame(X_train_s, columns=feature_names),
    y_train.values.ravel(),
    eval_set=[(pd.DataFrame(X_test_s, columns=feature_names), y_test.values.ravel())],
)

print("✅ LightGBM trained!")

✅ LightGBM trained!


D:\Amin\My Python Projects\MFT-Kafka-Ensemble-Optimization\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


In [16]:
y_pred_lgbm = lgbm_model.predict(pd.DataFrame(X_test_s, columns=feature_names))

true = y_test.values[:, 0]

print(f"\nTHROUGHPUT — LightGBM")
print(f"  R²   : {r2_score(true, y_pred_lgbm):.4f}")
print(f"  MAE  : {mean_absolute_error(true, y_pred_lgbm):.2f} docs/sec")
print(f"  RMSE : {np.sqrt(mean_squared_error(true, y_pred_lgbm)):.2f} docs/sec")


THROUGHPUT — LightGBM
  R²   : 0.8556
  MAE  : 7314.17 docs/sec
  RMSE : 10667.87 docs/sec


In [17]:
from itertools import product

In [18]:
def recommend_lgbm(bill_size, top_n=3):
    combos = list(product(BATCH_SIZE_VALUES, LINGER_MS_VALUES, COMPRESSION_VALUES))
    grid   = pd.DataFrame(combos, columns=["batchSize", "lingerMs", "compressionType"])
    grid["billSize"] = bill_size

    X_grid = preprocessor.transform(grid)
    preds  = lgbm_model.predict(X_grid)
    grid["predicted_throughput"] = preds

    grid = grid.sort_values("predicted_throughput", ascending=False)

    print(f"\n{'='*50}")
    print(f"LightGBM  |  billSize = {bill_size:,}")
    print(f"{'='*50}")
    for rank, (_, row) in enumerate(grid.head(top_n).iterrows(), 1):
        print(f"\nRank #{rank}")
        print(f"  batchSize       : {int(row['batchSize'])}")
        print(f"  lingerMs        : {int(row['lingerMs'])} ms")
        print(f"  compressionType : {row['compressionType']}")
        print(f"  → throughput    : {row['predicted_throughput']:,.0f} docs/sec")

In [19]:
recommend_lgbm(bill_size=309000)
recommend_lgbm(bill_size=618000)
recommend_lgbm(bill_size=927000)


LightGBM  |  billSize = 309,000

Rank #1
  batchSize       : 65536
  lingerMs        : 385 ms
  compressionType : lz4
  → throughput    : 172,550 docs/sec

Rank #2
  batchSize       : 65536
  lingerMs        : 425 ms
  compressionType : lz4
  → throughput    : 172,538 docs/sec

Rank #3
  batchSize       : 65536
  lingerMs        : 345 ms
  compressionType : lz4
  → throughput    : 171,630 docs/sec

LightGBM  |  billSize = 618,000

Rank #1
  batchSize       : 65536
  lingerMs        : 445 ms
  compressionType : none
  → throughput    : 156,570 docs/sec

Rank #2
  batchSize       : 65536
  lingerMs        : 345 ms
  compressionType : none
  → throughput    : 156,082 docs/sec

Rank #3
  batchSize       : 65536
  lingerMs        : 425 ms
  compressionType : none
  → throughput    : 155,839 docs/sec

LightGBM  |  billSize = 927,000

Rank #1
  batchSize       : 65536
  lingerMs        : 85 ms
  compressionType : none
  → throughput    : 123,851 docs/sec

Rank #2
  batchSize       : 49152
  

D:\Amin\My Python Projects\MFT-Kafka-Ensemble-Optimization\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
D:\Amin\My Python Projects\MFT-Kafka-Ensemble-Optimization\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
D:\Amin\My Python Projects\MFT-Kafka-Ensemble-Optimization\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [20]:
import joblib

In [21]:
joblib.dump(lgbm_model, str(MODELS_DIR / "lgbm_model.pkl"))

['D:\\Amin\\My Python Projects\\MFT-Kafka-Ensemble-Optimization\\artifacts\\models\\lgbm_model.pkl']

In [22]:
import joblib
import numpy as np

# ذخیره داده های پردازش شده
joblib.dump(X_train_s,    str(SPLITS_DIR / "X_train_s.pkl"))
joblib.dump(X_test_s,     str(SPLITS_DIR / "X_test_s.pkl"))
joblib.dump(y_train,      str(SPLITS_DIR / "y_train.pkl"))
joblib.dump(y_test,       str(SPLITS_DIR / "y_test.pkl"))
joblib.dump(y_train_s,    str(SPLITS_DIR / "y_train_s.pkl"))
joblib.dump(y_test_s,     str(SPLITS_DIR / "y_test_s.pkl"))
joblib.dump(target_scaler,str(PREPROCESSING_DIR / "target_scaler.pkl"))
joblib.dump(preprocessor, str(PREPROCESSING_DIR / "preprocessor.pkl"))

print("✅ همه داده ها ذخیره شدند!")

✅ همه داده ها ذخیره شدند!
